# Make a class iterable or container-like

You want your class to work with `for x in my_obj`, `len(my_obj)`, `x in my_obj`, and maybe `my_obj[i]`. All of these are separate dunder methods, and you can pick and choose — implement the ones that make sense for your type.

This recipe walks through each one in turn, then builds a complete container class combining them. For the underlying concepts, see the [iterators and generators guide](../../iterators-and-generators/).

## `__iter__` — make `for x in obj` work

The simplest path: implement `__iter__` to return an iterator. Usually the iterator is just the internal list's own iterator:

In [ ]:
class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __iter__(self):
        return iter(self.songs)

p = Playlist(["Prelude", "Interlude", "Finale"])
for song in p:
    print(song)

Returning `iter(self.songs)` gives you a fresh iterator each time. That matters: if you iterate the same Playlist twice, both loops start from the beginning. Returning `self` and implementing `__next__` is a different pattern — it makes the object itself single-use, which is usually not what you want. See the [iterators and generators guide](../../iterators-and-generators/) for when it is.

## `__len__` — make `len(obj)` work

Implementing `__len__` also gives you free truthiness: empty `Playlist`s are falsy, non-empty ones are truthy. See the [truthiness rules](../../conditional-logic/reference/truthiness-rules.md).

In [ ]:
class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __iter__(self):
        return iter(self.songs)

    def __len__(self):
        return len(self.songs)

empty = Playlist([])
loaded = Playlist(["A", "B"])
print(len(loaded), bool(empty), bool(loaded))

## `__contains__` — make `x in obj` work

Without `__contains__`, Python falls back to iterating with `__iter__` and comparing each element. That works, but if there's a faster path — a set lookup, say — implement `__contains__` directly.

In [ ]:
class Inventory:
    def __init__(self, items):
        self.items = list(items)
        self._set = set(items)   # fast membership test

    def __iter__(self):
        return iter(self.items)

    def __contains__(self, item):
        return item in self._set

inv = Inventory(["apple", "pear", "plum"])
print("pear" in inv, "grape" in inv)

## `__getitem__` — make `obj[i]` work

Implementing `__getitem__` is what gives you indexing and slicing. A nice side effect: Python will treat your class as iterable *even without `__iter__`*, by calling `__getitem__(0)`, `(1)`, `(2)` until it gets `IndexError`. That's the old iteration protocol; it still works. (Most classes define both for clarity.)

In [ ]:
class Song:
    def __init__(self, title):
        self.title = title
    def __repr__(self):
        return f"Song({self.title!r})"

class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __getitem__(self, index):
        return self.songs[index]   # works for int AND slice

    def __len__(self):
        return len(self.songs)

p = Playlist([Song("A"), Song("B"), Song("C")])
print(p[0])
print(p[-1])
print(p[::2])      # slicing works because list slicing does

Because `self.songs[index]` handles both integers and slices, slicing just works. If you were implementing `__getitem__` from scratch (not delegating to a list), you'd need to check `isinstance(index, slice)` and handle the two cases separately.

## Putting it all together

A complete container class — iterable, sized, membership-testable, indexable, and with a sensible `__repr__`:

In [ ]:
class Playlist:
    def __init__(self, songs=()):
        self._songs = list(songs)

    def add(self, song):
        self._songs.append(song)

    def __iter__(self):
        return iter(self._songs)

    def __len__(self):
        return len(self._songs)

    def __contains__(self, song):
        return song in self._songs

    def __getitem__(self, index):
        return self._songs[index]

    def __repr__(self):
        return f"Playlist({self._songs!r})"

p = Playlist(["Prelude", "Interlude", "Finale"])
p.add("Encore")
print(p)
print(len(p))
print("Finale" in p)
print(p[0])
print(list(p))

## Shortcut: `collections.abc` base classes

If you want the full suite of container methods (`index`, `count`, `__contains__`, `__iter__`, `__reversed__`, and so on), you can inherit from one of the abstract base classes in `collections.abc` — `Sequence`, `MutableSequence`, `Mapping`, `Set`. Implement the handful of required methods and the ABC fills in the rest.

In [ ]:
from collections.abc import Sequence

class Playlist(Sequence):
    def __init__(self, songs=()):
        self._songs = list(songs)

    # Sequence requires __getitem__ and __len__; everything else is free.
    def __getitem__(self, index):
        return self._songs[index]

    def __len__(self):
        return len(self._songs)

p = Playlist(["Prelude", "Interlude", "Finale"])
print(p.index("Interlude"))        # Sequence.index, from the ABC
print(p.count("Prelude"))           # Sequence.count, from the ABC
print(list(reversed(p)))             # __reversed__ from the ABC

That's the quickest way to get a full container interface. For small classes where you only need `__iter__` and `__len__`, it's overkill — the previous approach reads more directly.